Más análisis exploratorio de los datos...

In [63]:
import pandas as pd 
import numpy as np

In [ ]:
catalogo_productos = pd.read_csv("Datos-finales/catalogo_productos.csv")
especificaciones_cajas = pd.read_csv("Datos-finales/especificaciones_cajas.csv")
operaciones_planta = pd.read_csv("Datos-finales/operaciones_planta.csv")
procurement_cajas = pd.read_csv("Datos-finales/procurement_cajas.csv")

Agregamos las cargas máximas para cada tipo de caja (pasamos de mm a m)

In [ ]:
#Calculo el perímetro. Divido por 1000 para pasar a m
perimetro_cajas = 2 * (especificaciones_cajas["caja_exterior_largo"] + especificaciones_cajas["caja_exterior_ancho"])/ 1000 

ect_por_grosor = {2.5: 600, 2.7: 730, 3.0: 1000, 4.1: 1200, 4.5: 1400, 4.6: 1450, 4.7: 1500, 4.8: 1550, 5.0: 1650}

ects = [ect_por_grosor[g] for g in especificaciones_cajas['caja_grosor_mm']]

especificaciones_cajas['carga_max'] = ects * perimetro_cajas / 9.81

Averigüemos si la carga soportada actualmente se acerca a la máxima

¿cuenta el peso del producto en la base de la pila?

In [189]:
merged = catalogo_productos.merge(
    especificaciones_cajas[['caja_tipo_id', 'cantidad_cajas_alto', 'carga_max', 'caja_grosor_mm',
                            'caja_exterior_largo','caja_exterior_ancho','caja_exterior_alto',
                            'caja_interior_largo','caja_interior_ancho','caja_interior_alto']],
    on='caja_tipo_id',
    how='left'
)
#El -1 es para NO contar la caja de la base
catalogo_productos['carga_soportada'] = merged['peso_neto_caja'] * (merged['cantidad_cajas_alto'] - 1)

In [190]:
catalogo_productos[catalogo_productos['carga_soportada']/ merged['carga_max'] > 1]

,codigo_producto,descripcion_producto,ingrediente_forma,tipo_proyecto,subcategoria,categoria,tamaño_corte,peso_neto_caja,tamaño_paquete,cantidad_paquetes,...,carga_soportada,produco_largo,produco_ancho,produco_alto,producto_largo,producto_ancho,producto_alto,headspace_largo,headspace_ancho,headspace_alto


El numero máximo de cajas que se pueden apilar para un producto dado es

$$n_{max\_carga} = \lfloor(carga\_max/peso\_neto\_paquete)\rfloor + 1 $$

mientras este número no haga que la pila sopbrepase los 1800 mm

In [191]:
merged[merged['caja_exterior_alto'] * merged['cantidad_cajas_alto'] >= 1800]

,codigo_producto,descripcion_producto,ingrediente_forma,tipo_proyecto,subcategoria,categoria,tamaño_corte,peso_neto_caja,tamaño_paquete,cantidad_paquetes,...,headspace_alto,cantidad_cajas_alto,carga_max,caja_grosor_mm,caja_exterior_largo,caja_exterior_ancho,caja_exterior_alto,caja_interior_largo,caja_interior_ancho,caja_interior_alto


In [192]:
merged['n_por_ancho_pallet'] = (1200 // merged['caja_exterior_ancho']).astype(int)   # 1200mm / ancho caja
merged['n_por_largo_pallet'] = (800 // merged['caja_exterior_largo']).astype(int)   # 800mm / largo caja
merged['n_capas_max_altura'] = (1800 // merged['caja_exterior_alto']).astype(int) # limitado por altura

In [193]:
merged['n_max_carga'] = ((merged['carga_max'] // merged['peso_neto_caja']) + 1).astype(int)

El n_max de cajas que se pueden apilar es el mínimo entre el número máximo que soporta el tipo de caja y el máximo número antes de que sobrepase los 1800 mm de altura

In [194]:
merged['n_max'] = np.minimum(
    merged['n_capas_max_altura'],
    merged['n_max_carga']
)

In [195]:
merged[merged['n_capas_max_altura'] <= merged['n_max_carga']].shape

(435, 36)

Siempre gana la altura! Se puede bajar la carga_max entonces... probemos con el mínimor grosor permitido = 3.0 mm

In [196]:
nuevo_per = 2 * (merged["caja_interior_largo"] + merged["caja_interior_ancho"] + 12.0)/ 1000
merged['max_carga_3mm'] = 1000 * nuevo_per / 9.81
merged['n_max_carga_3mm'] = ((merged['max_carga_3mm'] // merged['peso_neto_caja']) + 1).astype(int)

In [197]:
merged['n_capas_max_altura_3mm'] = (1800 // (merged['caja_interior_alto'] + 6.0)).astype(int)

In [198]:
merged[merged['n_capas_max_altura_3mm'] <= merged['n_max_carga_3mm']].shape

(435, 39)

Sigue ganando altura!

In [199]:
merged[merged['n_capas_max_altura_3mm'] > merged['n_capas_max_altura']].shape

(63, 39)

Además en algunos casos ganamos una capa más

In [200]:
merged['n_por_ancho_pallet_3mm'] = (1200 // (merged['caja_interior_ancho'] + 6.0)).astype(int)
merged['n_por_largo_pallet_3mm'] = (800 // (merged['caja_interior_largo'] + 6.0)).astype(int)

In [201]:
merged[merged['n_por_ancho_pallet_3mm'] == merged['n_por_ancho_pallet']].shape

(435, 41)

In [202]:
merged[merged['n_por_ancho_pallet_3mm'] == merged['n_por_ancho_pallet']].shape

(435, 41)

In [203]:
ids = merged[merged['n_por_ancho_pallet_3mm'] < merged['n_por_largo_pallet']]['caja_tipo_id']
print(ids.shape)
especificaciones_cajas[especificaciones_cajas['caja_tipo_id'].isin(ids)]['caja_grosor_mm'].value_counts()

(0,)


Series([], Name: count, dtype: int64)

En ancho no ganamos nada... en largo tampoco y de hecho "perdimos" aunque en realidad esos 5 son con grosor menor el cual no está permitido

Si asumimos que el volumen interno actual de las cajas ES el volumen del envase primario entonces:

In [204]:
catalogo_productos['producto_largo'] = merged['caja_interior_largo']
catalogo_productos['producto_ancho'] = merged['caja_interior_ancho']
catalogo_productos['producto_alto'] = merged['caja_interior_alto']

Y el headspace del producto se calcula como:

In [205]:
catalogo_productos['headspace_largo'] = merged['caja_interior_largo'] - catalogo_productos['producto_largo'] 
catalogo_productos['headspace_ancho'] = merged['caja_interior_ancho'] - catalogo_productos['producto_ancho'] 
catalogo_productos['headspace_alto'] = merged['caja_interior_alto'] - catalogo_productos['producto_alto'] 

El cual debe cumplir con una restricción dependiendo del grosor de la caja y con un tope absoluto de 40 mm:

In [ ]:
max_headspace = {2.5: 0.06, 2.7: 0.06, 3.0: 0.06, 4.1: 0.08, 4.5: 0.08, 4.6: 0.1, 4.7: 0.1, 4.8: 0.1, 5.0: 0.1}

headspace_cols = [c for c in catalogo_productos.columns if c.startswith('headspace')]

for c in headspace_cols:
    dim = c.replace('headspace_', '')
    col_name = f'max_headspace_{dim}'
    
    max_hs_por_caja = [max_headspace[g] for g in merged['caja_grosor_mm']]
    
    dim_int_caja = merged[f'caja_interior_{dim}']
    
    catalogo_productos[col_name] = dim_int_caja * max_hs_por_caja
    
    #El máximo es 40 mm
    catalogo_productos[catalogo_productos[col_name] > 40.0] = 40.0